# Temporal Patterns in ERCOT Day-Ahead Prices — HB_HOUSTON

I focused on a single settlement point — the Houston trading hub (`HB_HOUSTON`) — for three reasons: it’s one of the most liquid hubs in ERCOT, it sits in the load pocket that drives a lot of summer scarcity, and restricting to one series makes “hour of day” and “month of year” aggregations unambiguous (no double counting across the 12 location-series). Patterns here generalize qualitatively to the other hubs/zones, and the methodology is reusable.

## 1. Setup, subset, and spike label

We reuse the `$100/MWh` binary spike threshold settled on in notebook 01. We also define the meteorological season mapping (Dec–Feb = Winter, etc.) that the hour-of-day plot will split on.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ercot_merged_dataset.parquet'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 12,
})

SPIKE_THRESHOLD = 100.0

SEASON_MAP = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall',  10: 'Fall',  11: 'Fall',
}
SEASON_ORDER = ['Winter', 'Spring', 'Summer', 'Fall']
SEASON_COLORS = {
    'Winter': '#4c78a8',
    'Spring': '#54a24b',
    'Summer': '#e45756',
    'Fall':   '#f58518',
}

df = pd.read_parquet(DATA_PATH)
hh = df[df['Location'] == 'HB_HOUSTON'].copy()
hh['season'] = hh['month'].map(SEASON_MAP)
hh['is_spike'] = hh['SPP'] > SPIKE_THRESHOLD

print(f'HB_HOUSTON rows: {len(hh):,}')
print(f'Date range: {hh["datetime"].min()} -> {hh["datetime"].max()}')
print(f'Spike threshold: ${SPIKE_THRESHOLD:.0f}/MWh')
print(f'Spike hours: {hh["is_spike"].sum():,} ({hh["is_spike"].mean()*100:.2f}% of rows)')
print(f'\nHours per season:')
print(hh["season"].value_counts().reindex(SEASON_ORDER).to_string())

## 2. `price_by_hour.png` — hour-of-day profile by season

Electricity demand follows a strongly diurnal pattern, but the shape of that pattern shifts with season: summer is air-conditioning driven (afternoon peak), winter is heating driven (bimodal morning/evening peak in Texas’ gas-heated housing stock), and shoulder seasons look flatter. Plotting the four seasonal curves on one axis shows whether peak pricing hours track peak load hours, and whether the peak time-of-day migrates across the year.

In [ ]:
hourly_by_season = (
    hh.groupby(['season', 'hour'])['SPP']
      .mean()
      .unstack('hour')
      .reindex(SEASON_ORDER)
)

fig, ax = plt.subplots(figsize=(11, 6))
for season in SEASON_ORDER:
    ax.plot(hourly_by_season.columns, hourly_by_season.loc[season],
            marker='o', markersize=4, linewidth=2,
            color=SEASON_COLORS[season], label=season)
ax.set_xticks(range(0, 24))
ax.set_xlabel('Hour of day (local, America/Chicago)')
ax.set_ylabel('Mean Day-Ahead SPP ($/MWh)')
ax.set_title('HB_HOUSTON — average hourly price by season, 2019–2026')
ax.legend(title='Season', loc='upper left')

out_path = OUTPUTS_DIR / 'price_by_hour.png'
fig.tight_layout()
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()

Summer shows a single sharp afternoon peak (roughly hours 15–19) where AC load and limited reserves push prices well above the rest of the day. Winter has the expected bimodal shape — a morning ramp around hours 6–9 and a second evening peak around hours 18–20 — though the winter mean is pulled upward by Winter Storm Uri in Feb 2021; for most winter hours the two humps are modest. Spring and Fall sit below summer throughout the day and are the flattest seasons, which makes sense because both load and generation are easiest to balance in mild weather. So we know that the interaction with season is important, I should include a hour × month interaction terms (or a model flexible enough to learn them) rather than treating them as independent.

## 3. `price_by_month.png` — monthly distribution

A boxplot lets us see distributional shape month by month, not just the mean. I capped the displayed values at \$500/MWh — anything above becomes a fliers hint rather than a plotted dot. This keeps the interquartile boxes readable. 

In [ ]:
Y_CAP = 500.0

monthly_data = [hh.loc[hh['month'] == m, 'SPP'].clip(upper=Y_CAP).values for m in range(1, 13)]
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(11, 6))
bp = ax.boxplot(monthly_data, labels=month_names, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=3, markerfacecolor='#888',
                                markeredgecolor='none', alpha=0.4))
for patch in bp['boxes']:
    patch.set_facecolor('#a0c4e8')
    patch.set_edgecolor('#1f3b5b')
for median in bp['medians']:
    median.set_color('#1f3b5b')
    median.set_linewidth(1.5)

ax.set_ylim(-50, Y_CAP + 20)
ax.set_xlabel('Month')
ax.set_ylabel('Day-Ahead SPP ($/MWh)  —  capped at $500 for display')
ax.set_title('HB_HOUSTON — monthly price distribution (values > $500 clipped for readability)')

out_path = OUTPUTS_DIR / 'price_by_month.png'
fig.tight_layout()
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()

clip_stats = []
for m in range(1, 13):
    vals = hh.loc[hh['month'] == m, 'SPP']
    total = len(vals)
    clipped = (vals > Y_CAP).sum()
    clip_stats.append({
        'Month': month_names[m - 1],
        'Total Hours': total,
        'Clipped (>$500)': clipped,
        'Clipped %': f'{100 * clipped / total:.2f}%' if total > 0 else '0%',
        'Mean (raw)': f'${vals.mean():.2f}',
        'Mean (capped)': f'${vals.clip(upper=Y_CAP).mean():.2f}',
    })

clip_df = pd.DataFrame(clip_stats)
print(clip_df.to_string(index=False))

Summer months (June–September) have visibly higher medians and much wider IQRs than the shoulder seasons, consistent with AC-driven scarcity. February’s fliers cluster high — Winter Storm Uri is visible as a dense band of above-cap observations. The clipped-share table printed below the plot quantifies how much of each month got capped. We can see that February is the clear winter outlier despite low overall winter means, because extreme events in Texas winters are concentrated in a small number of extremely cold hours rather than spread across the season.

## 4. `price_heatmap.png` — mean price by (hour, month)

In [ ]:
price_grid = (
    hh.groupby(['hour', 'month'])['SPP']
      .mean()
      .unstack('month')
      .reindex(index=range(24), columns=range(1, 13))
)

overall_mean = hh['SPP'].mean()
half_range = max(overall_mean - price_grid.min().min(), price_grid.max().max() - overall_mean)
vmin = overall_mean - half_range
vmax = overall_mean + half_range

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(price_grid.values, aspect='auto', origin='lower',
               cmap='RdBu_r', vmin=vmin, vmax=vmax)
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(0, 24))
ax.set_xlabel('Month')
ax.set_ylabel('Hour of day')
ax.set_title(f'HB_HOUSTON — mean Day-Ahead SPP by hour × month\n(diverging scale centered on overall mean = ${overall_mean:.2f}/MWh)')
cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label('Mean SPP ($/MWh)')
ax.grid(False)

out_path = OUTPUTS_DIR / 'price_heatmap.png'
fig.tight_layout()
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path.relative_to(PROJECT_ROOT)}')

The hottest cells (red) cluster in the summer-afternoon corner (June–September, hours 14–20). February has a visible warm band across most hours, again driven by Uri. The fact that the diverging pattern persists even after averaging across six+ years of data tells us the seasonal-diurnal structure is a stable phenomenon across years. 

## 5. `spike_frequency_heatmap.png` — spike rate by (hour, month)

Average price and spike *frequency* aren’t the same thing: a cell can have a moderate mean while still producing disproportionate numbers of spike hours, or vice versa. Since the modeling task is specifically spike prediction, this is the view that most directly tells us where the positive class lives. We compute, for each (hour, month) cell, the share of hours where `SPP > $100/MWh`.

In [ ]:
spike_grid = (
    hh.groupby(['hour', 'month'])['is_spike']
      .mean()
      .mul(100)
      .unstack('month')
      .reindex(index=range(24), columns=range(1, 13))
)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(spike_grid.values, aspect='auto', origin='lower',
               cmap='YlOrRd', vmin=0)
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(0, 24))
ax.set_xlabel('Month')
ax.set_ylabel('Hour of day')
ax.set_title(f'HB_HOUSTON — spike rate by hour × month\n(spike = SPP > ${SPIKE_THRESHOLD:.0f}/MWh; cell value = % of hours classified as spikes)')
cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label('Spike rate (%)')
ax.grid(False)

out_path = OUTPUTS_DIR / 'spike_frequency_heatmap.png'
fig.tight_layout()
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path.relative_to(PROJECT_ROOT)}')

Compared to the mean-price heatmap the hotspots are sharper and more concentrated: most spike mass sits in July–September afternoons (hours ~15–20). February has a warm column across many hours that the mean-price plot also showed — that’s Uri dominating the monthly total. Early-morning hours essentially never spike regardless of month, which is the mirror image of where the red mass lives in the price plot. So `hour` and `month` (or a `season × hour` interaction) will be useful for the spike classifier, even before any weather/gas features are added.

## 6. `weekend_vs_weekday.png` — weekday vs weekend profile

Industrial and commercial load drops sharply on Saturdays and Sundays, which should translate into a softer afternoon peak. A simple 24-hour profile split by `is_weekend` makes this visible. We pair it with a second subplot showing spike frequency by hour for the same split — because, similar to the last section, mean price and spike rate don’t always move together.

In [ ]:
profile = (
    hh.assign(day_type=np.where(hh['is_weekend'], 'Weekend', 'Weekday'))
      .groupby(['day_type', 'hour'])
      .agg(mean_price=('SPP', 'mean'), spike_rate=('is_spike', 'mean'))
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)

for day_type, color in [('Weekday', '#1f3b5b'), ('Weekend', '#e45756')]:
    s = profile.loc[day_type]
    ax1.plot(s.index, s['mean_price'], marker='o', markersize=4, linewidth=2,
             color=color, label=day_type)
    ax2.plot(s.index, s['spike_rate'] * 100, marker='o', markersize=4, linewidth=2,
             color=color, label=day_type)

ax1.set_xticks(range(0, 24, 2))
ax1.set_xlabel('Hour of day')
ax1.set_ylabel('Mean Day-Ahead SPP ($/MWh)')
ax1.set_title('Average hourly price')
ax1.legend(loc='upper left')

ax2.set_xticks(range(0, 24, 2))
ax2.set_xlabel('Hour of day')
ax2.set_ylabel(f'Spike rate (% of hours > ${SPIKE_THRESHOLD:.0f}/MWh)')
ax2.set_title('Spike frequency')
ax2.legend(loc='upper left')

fig.suptitle('HB_HOUSTON — weekday vs weekend profile, 2019–2026', fontsize=14, y=1.02)

out_path = OUTPUTS_DIR / 'weekend_vs_weekday.png'
fig.tight_layout()
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path.relative_to(PROJECT_ROOT)}')

**Reading the two panels.** On the left, weekday and weekend curves have similar shapes but the weekday afternoon peak is visibly higher — commercial/industrial load lifts mid-day prices on Mon–Fri. On the right, the spike-rate gap is proportionally even larger: weekend afternoons produce noticeably fewer spike hours than weekday afternoons, even though the underlying weather is similar. This makes sense: spikes are a *function of load* conditional on weather: the same 100°F Saturday afternoon puts less stress on the grid than the same 100°F Tuesday afternoon, so scarcity-pricing events are less likely. `is_weekend` (already in the merged dataset) is therefore a meaningful feature for the spike classifier.

---

## Takeaways for modeling

1. **Hour × season interactions** Peak time-of-day shifts materially between summer (afternoon) and winter (bimodal)
2. **February is unique.** Uri shows up in every monthly view as a disproportionate tail contributor. When we train, we should both (a) keep February in the data (it’s the most spike-rich month per hour) and (b) be explicit about evaluating on years *without* Uri to understand how well the model generalizes off-distribution.
3. **Spikes occur most frequently at certain times** July–September, hours 15–20, weekdays — that’s where the positive class lives. The spike classifier baseline can probably be surprisingly good from calendar features alone; weather/gas features should improve predictions outside of these times.
4. **Weekday/weekend is an additional feature.** Already in the merged dataset, already informative on both price and spike rate.